# Tableau

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/tableau.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Tableau is a visual analytics platform: you model a data source, build worksheets by dragging fields onto shelves, and publish governed dashboards with extracts, schedules, and permissions.

## How should data be modeled for Tableau dashboards?

Mental model: the data source is a contract about grain, and a measure is only correct at the grain of the rows the model produces. Tableau aggregates whatever rows survive the joins; it has no way to know that a sum is now double counting. Write the grain next to every published source, for example "one row per order line per day".

Worked example: `fact_sales` is one row per line, so `SUM([net_revenue])` is valid. Join `order_items` on `order_id` and each line repeats once per item, inflating the sum. Fix it in the model by aggregating the many side to one row per order before the join, or wrap the measure as `{FIXED [order_id] : SUM([net_revenue])}` and aggregate that. Common mistake: verifying correctness on a small sample where the fan-out happens not to appear.

Tableau performs best when connected to clean, business-ready datasets at the right grain. Use warehouse marts, extracts, or semantic views that already define measures and dimensions clearly. Avoid pushing complex, inconsistent business logic into many separate workbooks.

Example:

```text
fact_sales_daily + dim_product + dim_region -> certified Tableau data source
```

Modeling rules that survive an audit:

+ Publish one certified data source per domain instead of dozens of workbook-level connections.
+ Define the grain explicitly. If `fact_sales` is one row per order line, every measure must be valid at that grain.
+ Keep joins in the warehouse where possible. Relationship-level modeling is easier to govern than ad-hoc joins inside workbooks.
+ Expose conformed dimensions (`dim_date`, `dim_customer`) so date logic and customer logic are identical everywhere.

```sql
-- The kind of mart Tableau should connect to: already conformed at a known grain
CREATE OR REPLACE VIEW analytics.mart_sales_daily AS
SELECT
  d.date_key,
  d.date_day,
  p.category,
  p.subcategory,
  r.region,
  r.country,
  SUM(f.net_revenue)  AS net_revenue,
  SUM(f.quantity)     AS units,
  COUNT(DISTINCT f.order_id) AS orders
FROM analytics.fact_sales f
JOIN analytics.dim_date d     ON d.date_key = f.date_key
JOIN analytics.dim_product p  ON p.product_key = f.product_key
JOIN analytics.dim_region r   ON r.region_key = f.region_key
GROUP BY 1, 2, 3, 4, 5, 6;
```

This improves performance and makes metrics easier to govern. Trade-off: a pre-aggregated mart is fast and consistent but limits drill-through to detail; keep the line-level fact available as a second (`Detail`) data source for analysts who need it, rather than hiding detail behind a slow live join.

## What is the difference between live connections and extracts in Tableau?

How each mode works. Live: Tableau sends a query to the source on every view change and renders the rows it returns. Nothing is stored, and source latency, concurrency limits, and warehouse permissions all apply. Extract: Tableau runs the source query once, encodes the result into a `.hyper` file, and every later view is answered by the local Hyper engine. Live trades speed for freshness; an extract trades freshness for speed and a refresh dependency.

What an extract stores: the materialized rows at extract time after extract filters, the column schema and data types, calculated field definitions, default field properties, and Hyper's dictionary- and run-length-encoded columnar storage. It does not store the source's indexes, views, or permissions, and it cannot see rows an extract filter removed. Common mistake: assuming an extract inherits warehouse row-level security. It does not. If access is enforced in the warehouse, stay live or replicate the rule as a Tableau user filter.

A live connection queries the source system at view time, which keeps data fresh but depends on source performance. An extract stores a snapshot optimized for Tableau, improving performance and reducing source load but requiring refresh management.

Example choice:

+ live: operational dashboard needing current data
+ extract: executive dashboard over stable daily aggregates

```text
Decision rule
  freshness need < extract refresh interval?  -> live (or shorten the schedule)
  source cannot tolerate dashboard concurrency? -> extract
  dashboard scans billions of rows? -> extract or pre-aggregate, never live
  row-level security implemented in the warehouse? -> live (so policies apply)
```

The choice depends on freshness needs, source capacity, dashboard speed, and governance.

```bash
# tabcmd: refresh an extract on demand after the nightly warehouse load
tabcmd login -s https://tableau.example.com -t mysite -u svc_tableau -p "$TABLEAU_PASSWORD"
tabcmd refreshextracts --project "Finance" --workbook "Revenue_v3" --datasource "Certified Sales"
tabcmd logout
```

Trade-off: extracts duplicate storage and add a scheduling dependency; live connections push load onto the warehouse and can be throttled by connection limits. A common compromise is a "hot" extract for executive aggregates refreshed hourly and live access for drill-through.

## What is data visualization in Tableau?

Mechanics: a view is a mapping from data to visual channels. Position is the most accurate channel, then length, then color, then size, then shape. That ordering is why a bar chart beats a pie for comparison. Choose the mark from the question: bars compare categories, lines show change over time, scatter shows correlation, heatmaps summarize two dimensions. Common mistake: encoding one measure on both color and length, so the eye counts the same value twice.

Data visualization is a way to represent data that is visually appealing and interactive. With advancements in technology, the number of business intelligence tools has increased which helps users understand data, data sets, data points, charts, graphs, and focus on its impact rather than understanding the tool itself.

Tableau's value is the speed of the visual loop: encode a question as a view, change a field's role or aggregation, and see the answer. A good visualization encodes one measure with position (bar/line), supports comparison with a shared axis, and avoids dual encoding that misleads (for example, encoding the same metric with both color and length).

## What is the difference between various BI tools and Tableau?

The durable distinction is where the semantic layer lives. Looker and dbt keep metric definitions in versioned code that applications can reuse. Power BI keeps them in a tabular model. Tableau keeps most of them in workbooks and published data sources, which are fast to author but harder to diff. That is the argument for letting a certified published source, not a workbook, own the definition of margin or revenue.

The basic difference between the traditional BI tools and Tableau lies in the efficiency and speed.
The architecture of Traditional BI tools has hardware limitations. While Tableau does not have any sort of dependencies
The traditional BI tools work on complex technologies while Tableau uses simple associative search to make it dynamic.
Traditional BI tools do not support multi-thread, in-memory, or multi-core computing while Tableau supports all these features after integrating complex technologies.
Traditional BI tools have a pre-defined data view while Tableau does a predictive analysis for business operations.

A fairer comparison in an interview: legacy tools (Cognos, MicroStrategy) were designed around server-side cubes and governed report bursting, while Tableau was designed around a client-side visual query model. Modern Tableau Server/Cloud has caught up on governance (permissions, extract schedules, data sources), and competitors like Power BI and Looker compete on price, semantic modeling, or embedded analytics rather than on drag-and-drop speed alone.

## What are different Tableau products?

Architecture: Desktop authors XML (`.twb`/`.tds`). Server and Cloud run VizQL to render views, plus the gateway, the extract scheduler, and a repository that stores content and permissions. Tableau Bridge keeps on-premises sources reachable from Cloud. Common mistake in a sizing conversation: quoting user count alone. Capacity also depends on concurrent views, extract count and size, and the refresh window.

Tableau like other BI tools has a range of products:
Tableau Desktop: Desktop product is used to create optimized queries out from pictures of data. Once the queries are ready, you can perform those queries without the need to code. Tableau desktop encompasses data from various sources into its data engine and creates an interactive dashboard.
Tableau Server: When you have published dashboards using Tableau Desktop, Tableau servers help in sharing them throughout the organization. It is an enterprise-level feature that is installed on a Windows or Linux server.
Tableau Reader: Tableau Reader is a free feature available on Desktop that lets you open and views data visualizations. You can filter or drill down the data but restricts editing any formulas or performing any kind of actions on it. It is also used to extract connection files.
Tableau Online: Tableau online is also a paid feature but doesn't need exclusive installation. It comes with the software and is used to share the published dashboards anywhere and everywhere.
Tableau Public: Tableau public is yet another free feature to view your data visualizations by saving them as worksheets or workbooks on Tableau Server.

Add Tableau Prep (visual data preparation), Tableau Cloud (the SaaS successor to Tableau Online), and the Tableau REST API/Metadata API for automation. In interviews, note the licensing split: Creator (author + publish), Explorer (edit on published sources), and Viewer (consume), because license mix is usually the largest Tableau cost line.

## What is a parameter in Tableau?

Mechanics: a parameter is a single typed value stored in the workbook, not a column. It is evaluated on the client and substituted into calculations, filters, reference lines, bins, and top-N logic. A parameter action sets it from a selection in a view. Because it is a scalar it cannot vary per row, so it cannot act as a per-row predicate.

Worked example: a `[Top N]` parameter feeds `RANK(SUM([Net Revenue])) <= [Top N]`, and changing it re-runs the view. Common mistake: using a parameter where a filter belongs. A parameter-driven filter often returns more rows than a normal filter, which is expensive on a live connection. Prefer a real filter unless the user must type or pick the value.

The parameter is a variable (numbers, strings, or date) created to replace a constant value in calculations, filters, or reference lines. For example, you create a field that returns true if the sales are greater than 30,000 and false if otherwise. Parameters are used to replace these numbers (30000 in this case) to dynamically set this during calculations. Parameters allow you to dynamically modify values in a calculation. The parameters can accept values in the following options:
All: Simple text field
List: List of possible values to select from
Range: Select values from a specified range

```text
// Parameter: [Revenue Threshold] (integer, default 30000)
// Calculated field: Above Threshold
[Net Revenue] >= [Revenue Threshold]
```


```text
// Dynamic top-N using a parameter instead of hard-coded 10
// Parameter: [Top N] (integer, default 10)
RANK(SUM([Net Revenue])) <= [Top N]
```

Trade-off: parameters are workbook-scoped and computed client-side, so a parameter-driven filter cannot be pushed down to the warehouse as efficiently as a normal filter. For large extracts, prefer a filter over a parameter where both would work.

## Tell me something about measures and dimensions?

Mechanics: role is what a field means to the query and type is how it is stored. A field's default aggregation decides the generated SQL: `SUM`, `AVG`, `COUNTD`, or `None` for row level. Dimensions become the GROUP BY, and measures are aggregated inside it. Common mistake: leaving `Customer ID` as a numeric measure, which produces a meaningless sum; set it to a dimension or use `COUNTD`. A second mistake is using `ATTR()` to hide a fan-out: it returns `*` when more than one value exists rather than picking one.

In Tableau, when we connect to a new data source, each field in the data source is either mapped as measures or dimensions. These fields are the columns defined in the data source. Each field is assigned a dataType (integer, string, etc.) and a role (discrete dimension or continuous measure).
Measures contain numeric values that are analyzed by a dimension table. Measures are stored in a table that allows storage of multiple records and contains foreign keys referring uniquely to the associated dimension tables.
While Dimensions contain qualitative values (name, dates, geographical data) to define comprehensive attributes to categorize, segment, and reveal the data details.

In practice: dimensions set the level of detail (the GROUP BY), and measures are aggregated within that level. Numeric fields can be used as dimensions (`Customer ID` should be a dimension, not a summed measure), and text/date fields can drive measures (`COUNTD`). Every measure needs a default aggregation (SUM, AVG, COUNTD, MIN/MAX, or `None` for row-level).

```sql
-- What Tableau sends for SUM(Net Revenue) by Category and Month
SELECT p.category, DATE_TRUNC('month', d.date_day) AS month, SUM(f.net_revenue) AS net_revenue
FROM fact_sales f
JOIN dim_product p ON p.product_key = f.product_key
JOIN dim_date d    ON d.date_key = f.date_key
GROUP BY 1, 2;
```

## What are continuous and discrete field types?

Mechanics: discrete (blue) fields create headers and partition the data; continuous (green) fields create an axis with a continuous scale. Dates are special: discrete `YEAR([Order Date])` gives one header per year, while the continuous form gives a timeline. Date part groups by month; date value keeps the exact date on an axis. Common mistake: rebuilding a calculation when the real fix is switching a pill from discrete to continuous.

Tableau's specialty lies in displaying data differently either in continuous format or discrete. Both of them are mathematical terms used to define data where continuous means without interruptions and discrete means are individually separate and distinct.
While the blue color indicates discrete behavior, the green color indicates continuous behavior. On one hand, the discrete view defines the headers and can be easily sorted, while continuous defines the axis in a graph view and cannot be sorted.
Discrete View Tableau

Rules of thumb: discrete (blue) fields create headers and partition the view; continuous (green) fields create an axis and interpolate. A date can be either — discrete `MONTH(date)` gives 12 headers, continuous `MONTH(date)` gives a timeline. Switching date from discrete to continuous (or vice versa) is one of the most common "why does my line chart look wrong" questions.

## What is aggregation and disaggregation of data?

Mechanics: with aggregate measures on, Tableau wraps the view in a GROUP BY and returns one row per dimension combination. With `Analysis > Aggregate Measures` off, it issues a row-level query and renders one mark per record, which can be millions. `COUNTD` needs a distinct pass; `SUM` does not.

Worked example: a view with 8 regions and 12 months returns 96 rows whether the source holds ten thousand rows or one hundred million, because the aggregation runs in the source. Turn aggregation off and the same view returns every row. Common mistake: confusing marks with source rows. Source rows drive query cost; marks drive render cost.

Aggregation of data means displaying the measures and dimensions in an aggregated form. The aggregate functions available in the Tableau tool are:
SUM (expression): Adds up all the values used in the expression. Used only for numeric values.
AVG (expression): Calculates the average of all the values used in the expression. Used only for numeric values.
Median (expression): Calculates the median of all the values across all the records used in the expression. Used only for numeric values.
Count (expression): Returns the number of values in the set of expressions. Excludes null values.
Count (distinct): Returns the number of unique values in the set of expressions.
Tableau, in fact, lets you alter the aggregation type for a view.

Disaggregation of data means displaying each and every data field separately.

Disaggregation matters because it changes the query and the number of marks: a disaggregated view returns one row per record (potentially millions of marks) while an aggregated view returns one row per dimension combination. Use `Analysis > Aggregate Measures` to toggle it, and prefer `ATTR()` only when you have proven the values are identical within the group.

```text
// Guard against fan-out double counting in a blended/joined model
// Measure: Net Revenue (correct regardless of join)
SUM([Net Revenue])
```


## What are the different types of joins in Tableau?

Mechanics: a physical join is written on the Data Source canvas and becomes part of every query. A relationship is logical: Tableau records the linking keys and decides at query time whether to join and at what grain, based only on the fields a view uses. Relationships default to many-to-many unless you set the cardinality, and they avoid fan-out when a measure comes from one side only. Blends are a separate client-side path.

Common mistake: adding a physical join to a relationship-based model and then joining the same table again. Tableau can combine them into a fan-out that inflates measures. Prefer the relationship, set its cardinality explicitly, and enforce uniqueness and foreign keys in the warehouse so the assumption is testable.

Tableau is pretty similar to SQL. Therefore, the types of joins in Tableau are similar:
Left Outer Join: Extracts all the records from the left table and the matching rows from the right table.
Right Outer Join: Extracts all the records from the right table and the matching rows from the left table.
Full Outer Join: Extracts the records from both the left and right tables. All unmatched rows go with the NULL value.
Inner Join: Extracts the records from both tables.

Tableau also adds a Left Join with "Assume Referential Integrity" optimization (see below), and Relationship-level "logical" joins that are not physical joins at all — Tableau decides at query time whether to join or blend based on the fields used. Distinguish physical joins (Data Source page, one SQL query) from relationships (logical model, per-view join resolution), because the performance and row-count consequences differ sharply.

```text
Physical join in Tableau
  orders (order_id, customer_id, amount)
  JOIN customers ON orders.customer_id = customers.customer_id
```


## Tell me the different connections to make with a dataset?

Mechanics: a connection is live or extract-based, and a data source uses a single table, multiple related tables, or custom SQL. Tableau pushes filters, aggregations, and joins to the source whenever it can. Common mistake: custom SQL blocks many of those optimizations, because the wrapper looks like an opaque subquery and filters are applied after it runs. Prefer a view or a certified mart, and keep custom SQL for what the connector cannot express.

There are two types of data connections in Tableau:
LIVE: Live connection is a dynamic way to extract real-time data by directly connecting to the data source. Tableau directly creates queries against the database entries and retrieves the query results in a workbook.
EXTRACT: A snapshot of the data, extract the file (.tde or .hyper file) contains data from a relational database. The data is extracted from a static source of data like an Excel Spreadsheet. You can schedule to refresh the snapshots which are done using the Tableau server. This doesn't need any connection with the database.

Modern extracts use the `.hyper` format (TDE is deprecated). Extracts can be full or incremental: an incremental refresh appends new rows using a "refresh if" condition, which is much cheaper than a full rebuild.

```text
Extract refresh options
  Full refresh          -> rebuild the whole snapshot (predictable, expensive)
  Incremental refresh   -> append rows where [Increment Key] > last refresh value
  Refresh from source   -> re-run the source query; then re-apply extract filters
```


## What are the supported file extensions in Tableau?

Mechanics: `.twb` and `.tds` are XML with no data; `.twbx` and `.tdsx` are zip packages that also carry extracts and assets. Common mistake: emailing a `.twb` that points at an extract sitting on the author's laptop, which breaks on the recipient's machine. Package it, or publish the data source and point the workbook at that.

The supported file extensions used in Tableau Desktop are:
Tableau Workbook (TWB): contains all worksheets, story points, dashboards, etc.
Tableau Data Source (TDS): contains connection information and metadata about your data source
Tableau Data Extract (TDE): contains data that has been extracted from other data sources.
Tableau Packaged Workbook (TWBX): contains a combination of the workbook, connection data, and metadata, and the data itself in the form of TDE.  It can be zipped and shared.
Tableau Packaged Data Source (TDSX): contains a combination of different files.
Tableau Bookmark (TBM): to earmark a specific worksheet.

Also `.hyper` (the current extract format, replacing `.tde`) and `.twbl` (Tableau workbook locale file). `.twb` does not contain extracts, so a `.twb` opened on another machine needs access to the same published data source; `.twbx` and `.tdsx` are self-contained packages and are what you attach to a support ticket.

## What are the supported data types in Tableau?

Mechanics: conversion happens either when the extract is built or inside the query. `INT`, `FLOAT`, `STR`, `DATE`, and `DATETIME` casts are valid, and a cast on a filter column usually stops the source from pruning. Common mistake: a numeric-looking string that never joins to a numeric key. Fix the type once upstream instead of casting it in every calculation.

The following data types are supported in Tableau:

| Data type | Example values |
|---|---|
| Boolean | True / False |
| Date | December 28, 2016 |
| Date & Time | December 28, 2016 06:00:00 PM |
| Geographical | Beijing, Mumbai (role = geographic) |
| String | "EMEA", "Germany" |
| Number (decimal) | 8.00 |
| Number (whole) | 5 |

Tableau also has a NULL type that appears when all values are unknown, and date/time values carry a time zone from the source. Aggregations on strings (`COUNTD`, `MIN`, `MAX`) are valid; arithmetic on strings is not, which is why a numeric-looking string column is a common extract-time fix.

## What are sets?

Mechanics: a constant set is a fixed member list; a computed set updates on refresh and can be condition-, top-N-, or formula-based. A set behaves as a boolean dimension (`IN`/`OUT`), can act as a filter, and can drive set actions. The in/out decision is computed at the aggregation grain of its condition, so it sees only filters that run before it.

Bins are the third bucketing tool. A bin splits a continuous measure into equal-width buckets and creates a new discrete dimension. Choose a width that matches the natural unit (for example 10 dollars) and keep the original measure for aggregation. Common mistake: binning a measure and then summing the bin field, which counts the same value twice.

Sets are custom fields created as a subset of the data in your Tableau desktop. Sets can be computed based on conditions or created manually based on the dimensions of the data source.
For example, A set of customers that earned revenue more than some value. Now, set data may update dynamically based on the conditions applied. Learn More

Sets are powerful because they can be combined with `IN`/`OUT`, used as a filter, and compared in a calculated field. A set with a cross-tab condition creates an in/out split that updates on refresh; a constant set does not. In/out sets also power set actions (click a mark to add it to the set), which is the cleanest way to build "highlight selected customers" dashboards.

```text
// Set: [Top 100 Customers]
// Condition: SUM([Net Revenue]) >= 100000, Top 100 by SUM([Net Revenue])
// Calculated field: [Is Top 100 Customer]
IF [Top 100 Customers] THEN 'Top 100' ELSE 'Long tail' END
```

## What are groups in Tableau?

Groups, sets, and bins compared: a group rewrites the members of one dimension into new labels and stays as metadata; a set marks members in or out and can be dynamic; a bin turns a continuous measure into buckets. Use groups for stable business rollups, sets for selection and comparison, and bins for distributions. Common mistake: building a group for something that changes, such as top-10 customers; that belongs in a computed set or in the warehouse.

Groups are created to visualize larger memberships using dimensions. Groups can create their own fields to categorize values in that specific dimension.

Groups are best for ad-hoc bucketing of a dimension (for example, mapping 60 country values into 6 regions). Groups are static metadata: they are not pushed into the warehouse query as a CASE expression, so both the members and the "Other" bucket are visible and reusable across worksheets.

```text
// Group: [Country (Group)] maps
//   Germany, France, Spain -> Europe
//   USA, Canada, Mexico    -> Americas
//   (everything else)      -> Other
```


## What are shelves?

Mechanics: Rows and Columns set the view's dimensions, marks encode measures, and the Detail shelf adds a dimension to the grain with no visual channel. Every field placed on Rows, Columns, Detail, or Color enters the GROUP BY, increasing the mark count and reducing how much the source can aggregate. Common mistake: dropping a high-cardinality field such as `Order ID` on Detail just to get a tooltip, which turns a fast aggregate into a row-level scan.

Tableau worksheets contain various named elements like columns, rows, marks, filters, pages, etc. which are called shelves. You can place fields on shelves to create visualizations, increase the level of detail, or add context to it.

+ Rows / Columns: the axes or headers; putting a dimension here changes the grain of the view.
+ Marks: color, size, label, detail, tooltip, path, shape — detail adds a level to the grain without visual encoding.
+ Filters: restrict rows returned by the query (dimension/measure/date/context filters live here).
+ Pages: split the view into a sequence of pages by a discrete field.

Performance note: every field on a shelf (especially on Marks > Detail and Color) adds to the GROUP BY, which increases the number of marks and can turn a fast aggregate into a slow high-cardinality view.

## Tell me something about Data blending in Tableau?

Mechanics: the primary source runs its query and defines the marks. The secondary source is queried separately, aggregated to the level of the linking dimension, and left-joined on the client. Only aggregated fields from the secondary are usable, and the link must be a dimension with a matching data type. Common mistake: expecting a secondary measure at a grain finer than the link, which is not available, or linking dates at different granularities, which silently yields nulls.

Worked example: primary `opportunities` at one row per opportunity; secondary `marketing_spend` summed per `account_id`. The blend shows spend beside each opportunity, but that spend repeats for every opportunity, so summing spend across opportunities double counts. If you need account-level spend at opportunity grain, land both in the warehouse and join on `account_id`.

Data blending is viewing and analyzing data from multiple sources in one place. Primary and secondary are two types of data sources that are involved in data blending.

Blending is a client-side left join: the primary source's marks are the anchor, and the secondary source is queried and matched on common linking fields. Because it happens after the primary query, the secondary source must be aggregated to the link level or it will not join.

```text
Primary:   CRM opportunities (opportunity_id, account_id, owner)
Secondary: Marketing spend (account_id, campaign, spend)
Link:      account_id
```


Rules: blend only when sources cannot be joined in the warehouse; the link field must have the same data type and can involve only one common dimension reliably; and a blended measure is available only at the link grain. If in doubt, land both sources in the warehouse and use a real join — blending is a fallback, not a modeling strategy.

## How do you generally perform load testing in Tableau?

Mechanics: every rendered view opens a VizQL session and runs a query, so load testing measures how many sessions the cluster sustains. `tabjolt` replays recorded user actions at a target concurrency, while the REST image endpoint is a lighter probe. Common mistake: benchmarking only with the REST image endpoint, which renders one static image and misses browser follow-up requests, parameter changes, and dashboard actions. Test at production concurrency and content.

Load testing in Tableau is done to understand the server's capacity with respect to its environment, data, workload, and use. It is preferable to conduct load testing at least 3-4 times in a year because with every new user, upgrade, or content authoring, the usage, data, and workload change.
Tabjolt was created by Tableau to conduct point-and-run load and performance testing specifically for Tableau servers. Tabjolt:
Automates the process of user-specified loads
Eliminates dependency on script development or script maintenance
Scales linearly with an increase in the load by adding more nodes to the cluster

Modern load testing is usually a mix of Tabjolt and the REST API. The table below maps the signal to the action:

| Symptom | Likely cause | Fix |
|---|---|---|
| Slow first render, fast refresh | heavy source query | extract or pre-aggregate |
| All views slow under load | server CPU/memory contention | more nodes, extract-heavy workload |
| Vizql timeouts at peak | too many concurrent live queries | caching, extract schedules, query throttling |
| Extract refresh misses window | source load or too-large extract | incremental refresh, split schedules |

```bash
# Simple concurrency probe against the REST API
for i in $(seq 1 50); do
  curl -s -o /dev/null -w "%{http_code} %{time_total}\n" \
    -H "X-Tableau-Auth: $TOKEN" \
    "https://tableau.example.com/api/3.19/sites/$SITE/views/$VIEW/image" &
done
wait
```

## Why would someone not use Tableau?

Mechanics of the cost: licensing is per Creator, Explorer, and Viewer seat plus Server or Cloud capacity, and extracts add storage and refresh compute. The budget usually goes to Creator seats for small edits or to a server sized for peak live concurrency. Common mistake: buying more capacity when the real fix is an extract, a pre-aggregated mart, or fewer marks on the view.

The limitations of using Tableau are:
Not cost-effective: Tableau is not that cost-effective when we compare it well with the other available data visualization tools. In addition to this, it has software upgrades, proper deployment, maintenance, and also training people for using the tool.
Not so secure: When it comes to data, everyone is extra cautious. Tableau focussed on security issues but fails to provide centralized data-level security. It pushes for row-level security and creates an account for every user which makes it more prone to security glitches.
BI capabilities are not enough: Tableau lacks basic BI capabilities like large-scale reporting, building data tables, or creating static layouts. It has limited result-sharing capabilities, email notification configuration is limited to admins, and the vendor doesn't support trigger-based notifications.

The fair version of this answer: Tableau is expensive per seat, its data-level security is implemented through user filters or row-level security in the source (both of which must be maintained), and it is not a scheduling/reporting engine for pixel-perfect operational documents. It is also weak as a semantic layer for programmatic consumers — Looker/dbt-style metrics and an API-first model are better when other applications need the same definitions. When NOT to use it: very high-volume operational reporting with strict formatting, or when the metric definitions must be consumed by services rather than humans.

## What is Tableau data engine?

Mechanics: Hyper is a columnar engine that stores an extract in a single file, uses dictionary and run-length encoding, and supports transactions and SQL. Tableau runs one Hyper process and queries it in-process, so an extract view avoids a network round trip. Hyper also executes cross-database joins and can be written directly through the Hyper API. Common mistake: treating the data engine as a cache. It is the execution engine over an extract; a live connection does not use it for the source query.

An analytical database that computes instant query responses, predictive analysis of the server, and integrated data. The data engine is useful when you need to create, refresh, or query extracts. It can be used for cross-database joins as well.

The data engine (Hyper) is a columnar in-memory/on-disk engine that executes against extracts. It is the reason `.hyper` extracts are fast: filters, aggregations, and cross-database joins run inside Hyper rather than round-tripping to the source.

```text
Source A (Oracle)  --\
                      >-- Hyper extract --(in-process query)--> view
Source B (Excel)   --/
```


## What are the various types of filters in Tableau?

The full order of operations, because it explains why a filter sometimes does not move a number:

```text
 1. Extract filters
 2. Data source filters
 3. Context filters
 4. Sets, top N, conditional filters, FIXED LOD
 5. Dimension filters
 6. INCLUDE / EXCLUDE LOD and data blending
 7. Measure filters
 8. Forecasts, table calculations, clusters, totals
 9. Table calculation filters
10. Trend lines and reference lines
```

What each type does. An extract filter drops rows before they are written into the `.hyper` file, so they are gone for every consumer of that extract. A data source filter applies to the source query and works for both live and extract connections. A context filter materializes a temporary, filtered data set, and every later filter becomes dependent on it; it is the only view filter a FIXED LOD respects. A dimension filter restricts discrete members after context and FIXED LOD. A measure filter applies after aggregation. A table calculation filter applies last, to the rendered result. Common mistake: applying a dimension filter to shrink a `{FIXED ...}` total. It will not change, because FIXED is computed before dimension filters. Promote the filter to a context filter instead.

Row-level security with user filters. `USERNAME()` returns the signed-in user, and `USERATTRIBUTE('region')` returns a site attribute set by an administrator. `[Region] = USERATTRIBUTE('region')` or the short-circuit `USERATTRIBUTE('region') = 'ALL' OR [Region] = USERATTRIBUTE('region')` scopes each viewer. A user filter runs in the query, so it can also prune rows on a live source. Common mistake: hard-coding a single shared service account for embedded analytics, which gives every viewer the same row set and defeats the filter entirely.

Tableau has 6 different types of filters:
Extract Filter: This filter retrieves a subset of data from the data source.
Dimension Filter: This filter is for non-aggregated data (discrete).
Data Source Filter: This filter refrains users from viewing sensitive information and thus reduces data feeds.
Context Filter: This filter creates datasets by applying presets in Tableau.
Measure Filter: This filter applies various operations like sum, median, avg, etc.
Table Calculation Filter: This filter is applied after the view has been created.

The order of operations is the interview point, because it explains why a filter does not remove a percentage:

```text
Extract filter -> Data source filter -> Context filter -> Dimension filter
   -> Measure filter -> Table calculation filter -> Percent-of-total table calc
```


FIXED LOD expressions are computed before dimension filters but after context filters — that ordering is why `{ FIXED ... }` totals can look "wrong" when a dimension filter is applied, and why context filters are used to constrain LODs.

## What are dual axes?

Mechanics: dual axis puts two measures on independent continuous scales and can mislead if the scales are unrelated. Combined axis stacks the measures on one shared scale, which is the honest choice for revenue and margin. Synchronizing the axes makes the two comparable but changes the visual message. Common mistake: dual-axing two series to force a correlation that the shared scale would not show, and leaving the right-hand axis unlabeled.

Dual axes are used to analyze two different measures at two different scales in the same graph. This lets you compare multiple attributes on one graph with two independent axes layered one above the other.
To add a measure as a dual-axis, drag the field to the right side of the view and drop it when you see a black dashed line appear. You can also right-click (control-click on Mac) the measure on the Columns or Rows shelf and select Dual Axis.

Distinguish dual axis (two independent scales, can mislead) from combined axis (one shared scale, honest comparison). For revenue and margin, a blended axis (`Measure Values`) on one scale is usually the correct choice; dual axis is for measures whose units genuinely differ (revenue in dollars and conversion rate in percent), and the axes should be labeled.

```text
// Dual axis best practice
Rows: SUM(Revenue)   -- synchronize axis OFF, label "USD"
Rows: SUM(Margin %)  -- synchronize axis OFF, label "%"
Format: label both axes; add a reference line at the target margin.
```

## What is the difference between a tree and heat map?

Mechanics: a heat map needs a dimension on rows, a dimension on columns, and a measure on color, and it is read by scanning intersections. A treemap needs a hierarchy and a size measure, and it is read by rectangle area. Both lose their message above roughly twenty categories, and a diverging palette should be reserved for deviations from a target, not magnitudes. Common mistake: using a rainbow palette on a sequential measure, which invents a boundary where the data is smooth.

Both the maps help in analyzing data. While a heat map visualizes and compares different categories of data, a treemap displays a hierarchical structure of data in rectangles. Heat map visualizes measures against dimensions by depicting them in different colors. Similar to a text table with values defined in different colors.
Heatmap In Tableau
Treemap visualizes the hierarchy of data in nested rectangles. Hierarchy levels are displayed from larger rectangles to smaller ones.
Example - Below treemap shows aggregated sales totals across a range of product categories:
TreeMap in Tableau

Practical differences: a heat map needs two dimensions (row/column) and a measure mapped to color, and is read by scanning intersections; a treemap needs a hierarchy and a size measure, and is read by rectangle area. Both degrade quickly above ~20 categories or with a non-sequential color palette, so use a diverging palette only for deviations (variance to target) and a sequential palette for magnitudes.

## What are extracts and schedules in Tableau server?

Mechanics: an extract refresh re-runs the source query and rewrites the `.hyper` file. A full refresh rebuilds everything. An incremental refresh appends rows that satisfy the extract's "Refresh if" condition, typically `[Order Date] > <last refresh value>` on a monotonically increasing key, so only new partitions are pulled. Common mistake: choosing a key that changes on existing rows, such as an `updated_at` that moves when a record is edited; incremental refresh then misses updates. Use a full refresh, or a key that only ever increases.

Data extracts are the subsets of data created from data sources. Schedules are scheduled refreshes made on extracts after publishing the workbook. This keeps the data up-to-date. Schedules are strictly managed by the server administrators.

```text
Schedule design pattern
  06:00  warehouse daily load completes
  06:15  full refresh: small dimension extracts (< 5 min)
  06:30  incremental refresh: large fact extracts (append today's rows)
  07:00  subscription emails + dashboard cache warm
```

Schedules run as the extract owner, so if that user is deactivated the refresh silently fails — always assign refreshes to a service account and monitor refresh failures through the REST API rather than trusting the UI status.

```bash
# Query extract refresh status (REST API), then alert on failures
curl -s -H "X-Tableau-Auth: $TOKEN" \
  "https://tableau.example.com/api/3.19/sites/$SITE/tasks/extractRefreshes" \
  | python -c "import sys,json; [print(t['id'], t.get('schedule',{}).get('frequency'), t.get('lastRunAt')) for t in json.load(sys.stdin)['extractRefreshes']['extractRefresh']]"
```

## What are the components in a dashboard?

Mechanics: dashboards compose worksheets through layout containers, and actions wire them together. A filter action passes a selection to other sheets, a highlight action emphasizes it, a parameter action sets a value, and a set action adds marks to a set. Dynamic zone visibility shows or hides a container from a parameter. Common mistake: many floating zones with fixed positions, which look right on the author's monitor and break on every other size. Use nested containers and device layouts.

The components displayed in a dashboard are:
Horizontal: Horizontal view allows the users to combine the worksheets and dashboard elements from left to right and edit the height of the elements.
Vertical: Vertical view allows the users to combine the worksheets and dashboard elements from top to bottom and edit the width of the elements.
Text: All the textual fields.
Image Extract: To extract an image Tableau applies some code, extracts the image, and saves it in a workbook in the XML format.
Web URL: Hyperlink that points to a web page, file, or other web resources outside of Tableau

Modern dashboards add layout containers (horizontal/vertical nesting), floating objects, device layouts (desktop/tablet/phone), dashboard actions (filter, highlight, URL, parameter, set, go-to-sheet), and dynamic zone visibility driven by a parameter. Layout containers are the difference between a dashboard that resizes gracefully and one that breaks on a laptop.

## What is a TDE file?

Mechanics: `.tde` is the legacy extract format and `.hyper` is its replacement. Tableau opens a `.tde` by converting it, but new extracts are always Hyper, and only Hyper is writable through the Hyper API. Common mistake in a migration: assuming a `.tde` can be edited in place. Rebuild it as `.hyper` and update the schedule, or the refresh quietly targets the old file.

TDE stands for Tableau Data Extract and uses the `.tde` extension. A TDE file points to a file that contains data from external sources like MS Excel, MS Access, or CSV files. TDE makes it easier to analyze and discover data.

TDE was replaced by the `.hyper` format; Tableau still opens `.tde` but converts it. For automation, use the Hyper API to write extracts directly without Tableau Desktop:

In [ ]:
# ILLUSTRATIVE - requires tableauhyperapi: install to run, or read on.
# Write a Hyper extract with the Hyper API (runs in a pipeline, no Desktop needed)
from tableauhyperapi import HyperProcess, Telemetry, Connection, CreateMode, TableDefinition

table = TableDefinition("Extract", [
    TableDefinition.Column("order_id", "text"),
    TableDefinition.Column("order_date", "date"),
    TableDefinition.Column("revenue", "double precision"),
])

with HyperProcess(telemetry=Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU) as hyper:
    with Connection(hyper.endpoint, "orders.hyper", CreateMode.CREATE_AND_REPLACE) as conn:
        conn.catalog.create_schema("Extract")
        conn.catalog.create_table(table)
        conn.execute_command(
            "COPY \"Extract\".\"Extract\" FROM 'orders.csv' WITH (format csv, header true)"
        )

## What is the story in Tableau?

Mechanics: a story is a sequence of captured worksheet states with captions. Each story point freezes the filters, selections, and sort of the moment it was captured. It is good for a one-off narrative such as a monthly business review. Common mistake: maintaining a story for a recurring review when a parameter-driven dashboard would update itself; every data change forces a manual re-capture.

Creating a story is effective in Tableau which is created by combining various charts to portray a plot of viewers. A story is a sheet that contains all the methods used to create those worksheets. To create a story:
Click the New Story on the dashboard.
Choose the right size of the story from the bottom-left corner or choose a custom size.
Start building the story by double-clicking the sheet and add it to the story point.
Add a caption to the story by clicking Add a caption.
You can update the highlights by clicking Update in the toolbar. You can also add layout options, format a story, or fit the story to your dashboard.

A story is a sequence of captured worksheet states with captions — useful for a narrative walkthrough (monthly business review), but every story point freezes the filters and selections of that moment, so it must be re-captured when the underlying data or a view changes. For recurring reviews, a parameter-driven dashboard is usually more maintainable than a hand-maintained story.

## What are different Tableau files?

Mechanics: `.twb` and `.tds` are XML and hold no data; `.twbx` and `.tdsx` are zip packages that bundle extracts and assets, which makes them portable but opaque to Git. `.hyper` is the extract itself, and `.tbm` bookmarks a single worksheet. Common mistake: committing a `.twbx` and losing the ability to review changes, since the archive is binary. Convert to `.twb` plus a published source for version control.

Workbooks: Workbooks contain one or more worksheets and dashboard elements.
Bookmarks: Contains a single worksheet that is easier to share.
Packaged Workbooks: Contains a workbook along with supporting local file data and background images.
Data Extraction Files: Extract files that contain a subset of data.
Data Connection Files: Small XML file with various connection information.

| File | Contains | Portable? | Typical use |
|---|---|---|---|
| `.twb` | worksheets, dashboards, XML | no data | version control in Git |
| `.twbx` | workbook + extracts + assets | yes | sharing a self-contained analysis |
| `.tds` | data source connection + metadata | no data | publish a shared data source |
| `.tdsx` | data source + extract | yes | share a source with an extract |
| `.hyper` | extract data | yes | scheduled extract, Hyper API |
| `.tbm` | single worksheet bookmark | no data | quickly reopen one view |

For version control, store `.twb`/`.tds` as XML in Git (never `.twbx`) so diffs are reviewable, and publish through a CI job rather than by hand.

## How do you embed views into webpages?

Mechanics: anonymous embedding allows access without a Tableau sign-in and must only be used for non-sensitive data. For authenticated embedding, trusted authentication exchanges a one-time ticket from the web server, and connected apps use a signed JWT so the viewer never sees a Tableau login. Common mistake: embedding with a hard-coded shared user, which makes every viewer look identical to row-level security.

You can easily integrate interactive views from your Tableau Server or Tableau online onto webpages, blogs, web applications, or internet portals. But to have a look at the views, the permissions demand the viewer to create an account on the Tableau Server. To embed views, click the Share button on the top of the view and copy the embed code to paste it on the web page.
You can also customize the embedded code or Tableau Javascript APIs to embed views.

For a public/anonymous embed, enable "Let people access this view without signing in" (and ensure the data is not sensitive). For a logged-in embed, use connected apps or trusted authentication so the user does not see a Tableau login page.

```text
Embed URL shape
https://tableau.example.com/views/Finance/Revenue?:showVizHome=no&:embed=y&:toolbar=no
&:tabs=no&:refresh=yes
```

In [ ]:
# ILLUSTRATIVE - requires requests: install to run, or read on.
# Trusted authentication: the web app asks Tableau Server for a one-time ticket
import requests

resp = requests.post(
    "https://tableau.example.com/trusted",
    data={"username": "viewer@acme.com", "client_ip": "10.0.0.5", "target_site": "finance"},
    timeout=10,
)
ticket = resp.text
url = f"https://tableau.example.com/trusted/{ticket}/views/Finance/Revenue?:embed=y"

Security note: embedding with a hard-coded user or a shared service account defeats row-level security. Always pass the real user identity through trusted auth/connected apps and let the warehouse or user filter enforce access.

## What is the maximum num of rows Tableau can utilize at one time?

Mechanics: there is no fixed row limit in Hyper; the practical ceilings are disk for the extract, memory during query execution, and render time for marks. Aggregation is pushed to the source or Hyper when possible, so the honest answer is that the limit is the number of marks and the shape of the source query, not a row count. Common mistake: quoting a row limit in an interview. Quote the two real limits and how you would reduce each.

The maximum number of rows or columns is indefinite because even though Tableau contains petabytes of data, it intelligently uses only those rows and columns which you need to extract for your purpose.

The real limits are practical: Hyper extract size is bounded by disk and by memory for query execution, marks are slower above ~100k, and cross-database joins run in the data engine. Tableau pushes aggregation to the source when possible, so the honest answer is "the limit is the number of marks and the source query, not a row count."

| Layer | Practical ceiling | Mitigation |
|---|---|---|
| Marks rendered | ~100k before interactivity suffers | aggregate, use density/heatmap |
| Extract size | tens to hundreds of GB | extract filters, partition, incremental refresh |
| Live query rows | source-dependent | pre-aggregate mart, materialized view |
| Columns in a view | dozens | hide unused fields, use folders |

## Mention what is the difference between published data sources and embedded data sources in Tableau?

Mechanics: a published data source is a standalone object with its own credentials, permissions, extract schedule, and refresh history; workbooks connect to it. An embedded source lives inside one workbook and duplicates credentials and refresh logic. Published sources are also where row-level security belongs, because a user filter defined once applies to every dependent workbook. Common mistake: rotating a database password in dozens of embedded sources instead of once in the published source.

Connection information is the details of data that you want to bring into Tableau. Before publishing it, you can create an extract of the same.
Published Data Source: It contains connection information that is independent of any workbook.
Embedded Data Source: It contains connection information which is connected to a workbook

Published sources are the governance mechanism: credentials are stored once, extract schedules run centrally, and a change to the source propagates to every dependent workbook. Embedded sources are convenient for ad-hoc work but mean every workbook owns its own connection and credentials rotation.

```bash
# Publish a data source over the REST API (initiate -> upload -> commit)
curl -s -X POST -H "X-Tableau-Auth: $TOKEN" -H "Content-Type: application/json" \
  -d '{"datasource":{"name":"Certified Sales","project":{"id":"'"$PROJECT_ID"'"}}}' \
  "https://tableau.example.com/api/3.19/sites/$SITE/datasources"
```

## What is the DRIVE Program Methodology?

Mechanics: DRIVE is a deployment methodology (Define, Refine, Implement, Validate, Expand) rather than a product feature. Its useful discipline is sequencing: agree the success metric and audience, pilot the model with a small group, add governance, measure adoption, then expand domain by domain. Common mistake: treating it as a checklist to complete rather than a loop that repeats per domain.

DRIVE program methodology creates a structure around data analytics derived from enterprise deployments. The drive methodology is iterative in nature and includes agile methods that are faster and effective.

DRIVE is Tableau's deployment methodology (Define, Refine, Implement, Validate, Expand). In interviews, connect it to a real rollout: define the success metric and audience first, refine the data model and pilot with a small group, implement governance (certified sources, permissions, extracts), validate adoption with usage telemetry, then expand domain by domain. The failure mode it prevents is a large launch with no adoption measurement.

## How to use groups in a calculated field?

Mechanics: a group created in the Data pane can be referenced by name in a calculation, because it is a field. An ad-hoc group made by selecting marks exists only in that view and cannot be referenced. If the grouping must be identical in the warehouse and in Tableau, express it once as a `CASE` expression in SQL and use that field. Common mistake: copying the group logic into several workbooks, which drift apart on the next change.

Add the GroupBy clause to SQL queries or create a calculated field in the data window to group fields.
Using groups in a calculation. You cannot reference ad-hoc groups in a calculation.
Blend data using groups created in the secondary data source: Only calculated groups can be used in data blending if the group was created in the secondary data source.
Use a group in another workbook. You can easily replicate a group in another workbook by copy and pasting a calculation.

A group created in the Data pane can be referenced by name in a calculation; an ad-hoc group created by selecting marks cannot. If you need the grouping logic in SQL, replicate it as a CASE expression so the warehouse and Tableau agree.

```sql
-- Warehouse-side equivalent of a Tableau group
SELECT
  CASE
    WHEN country IN ('Germany', 'France', 'Spain') THEN 'Europe'
    WHEN country IN ('USA', 'Canada', 'Mexico')   THEN 'Americas'
    ELSE 'Other'
  END AS region_group,
  SUM(net_revenue) AS net_revenue
FROM analytics.mart_sales_daily
GROUP BY 1;
```

## Explain when would you use Joins vs Blending in Tableau?

Three ways to combine data, and the right one depends on where the source of truth is. A join is physical and becomes part of every query. A relationship is logical: Tableau keeps the keys and chooses the join at query time from the fields used, so it can avoid fan-out. A blend is client-side: the primary query runs, the secondary is aggregated to the link, and the two are joined after the fact. Prefer a relationship, use a join when you want one fixed query shape, and blend only when the sources cannot be combined at all.

Worked example: orders in a warehouse table and a marketing spend CSV. Landing the CSV in the warehouse and relating it on `account_id` keeps everything in one query and one grain. Blending the CSV as a secondary source works, but the spend is only available at account grain and repeats across that account's orders, so any total over orders is wrong.

While the two terms may sound similar, there is a difference in their meaning and use in Tableau:
While Join is used to combine two or more tables within the same data source.
Blending is used to combine data from multiple data sources such as Oracle, Excel, SQL server, etc.

```text
Use a JOIN when:
  - both tables are in the same database
  - you need row-level detail from both sides
  - you want the work pushed down to the warehouse
```


Fan-out example: joining `orders` to `order_items` and then summing `orders.amount` double-counts the order for every item. The fix is to aggregate one side first or to use a FIXED LOD, not to blend.

## What is Assume referential integrity?

Mechanics: Assume Referential Integrity tells Tableau that every foreign key on the many side has a match on the one side, so the joined table can be omitted from a query when no field from it is used. It requires a left or right join, a unique or primary key on the one side, and a data source level join. Common mistake: turning it on to speed up a query where the assumption is false; unmatched fact rows then silently disappear, which is a correctness bug that looks like a performance win.

In some cases, you can improve query performance by selecting the option to Assume Referential Integrity from the Data menu. When you use this option, Tableau will include the joined table in the query only if it is specifically referenced by fields in the view.

Conditions: the join must be a left/right (not full outer) join, the join key on the "one" side must be unique or a primary key, and the join must be defined at the data source level. If the assumption is wrong — for example a fact row references a missing dimension row — rows silently disappear from the result, which is a correctness bug disguised as a performance win.

```sql
-- With Assume Referential Integrity, Tableau may generate this instead of a join:
SELECT SUM(amount) FROM fact_sales;                -- dimension not referenced
-- Without it, the engine must join to dim_product to prove the join is safe.
```

Use it deliberately for proven foreign keys, and add a warehouse-level not-null/foreign-key test to keep the assumption true.

## What is a Calculated Field and How Will You Create One?

Level of detail expressions control the grain at which an aggregate is computed. `FIXED` computes at the declared dimensions and ignores the view and all filters except extract, data source, and context filters. `INCLUDE` adds the declared dimensions to the view grain. `EXCLUDE` removes them from the view grain. The result is then aggregated to the view like any other measure.

```text
{ FIXED [Region] : SUM([Sales]) }            -- per region, whatever the view shows
{ INCLUDE [Customer Name] : SUM([Sales]) }   -- per customer, plus whatever is in the view
{ EXCLUDE [Region] : SUM([Sales]) }          -- total with region removed
{ FIXED : SUM([Sales]) }                     -- one value for the whole table
[Sales] / { FIXED : SUM([Sales]) }           -- share of total that survives a view filter
```

Row-level, aggregate, LOD, and table calculation compared. A row-level calculation runs once per row before aggregation: `[Sales] - [Cost]`. An aggregate calculation runs after the GROUP BY: `SUM([Sales]) - SUM([Cost])`. An LOD changes the grain the aggregate is computed at: `{ FIXED [Customer] : SUM([Sales]) }`. A table calculation runs last, on the values already in the view, and depends on its addressing: `RANK(SUM([Sales]))`. Moving a calculation between these layers changes its answer, not just its speed.

Worked example. Suppose East has 10 orders and 1000 revenue, West has 90 orders and 180000. `SUM(revenue) / SUM(orders)` is 1810 per order; `AVG(revenue / orders)` is 1050 per order. The first is a ratio of sums and is almost always the metric you mean; the second is an average of ratios and is only correct if each region should carry equal weight. Common mistake: building the ratio as `SUM([Sales]) / SUM([Orders])` in one sheet and as `AVG([Sales] / [Orders])` in another, then wondering why two dashboards disagree.

Calculated fields are created using formulas based on other fields. These fields do not exist but are created by you.
You can create these fields to:
Segment data
Convert the data type of a field, such as converting a string to a date.
Aggregate data
Filter results
Calculate ratios

There are three main types of calculations that you can create:
Basic Calculations: Transform values of the data fields at the source level
Level of Detail (LOD) Expressions: Transform values of the data fields at the source level like basic calculations but with more granular access
Table Calculations: Transform values of the data fields only at the visualization level
To create calculate fields:
In Tableau, navigate to Analysis>Create a calculated field. Input details in the calculation editor.
And, done!

Worked examples of each type:

```text
// Basic calculation (row-level, then aggregated by the view)
[Net Revenue] - [Cost]
```


Order of operations in one line: basic calculations at the data-source grain, LODs at the declared grain (after context filters), table calculations last, at the partition and addressing you choose. Trade-off: LODs run in the query and are expensive on large extracts; table calculations are cheap but only valid in the view they were built for and cannot be used as a dimension filter before computation.

## How Can You Display the Top Five and Bottom Five Sales in the Same View?

Mechanics: a top/bottom view is a filter on a rank, not a sort. Sorting only changes display order and leaves every mark present, so it cannot show exactly five. Set the rank with a table calculation (`RANK`, `RANK_DENSE`, `INDEX`) and filter on it, or use a top-N set filter, which is pushed into the query. Common mistake: a table calculation filter, which runs after the query and forces Tableau to fetch every customer before discarding all but five.

You can see top five and bottom five sales with the help of these functions:
Drag customer name to row and sales to the column.
Sort Sum(sales) in descending order.
Create a calculated field Rank of Sales.

```text
// Rank of Sales (table calculation, compute using Customer Name)
RANK_DENSE(SUM([Net Revenue]))
```


```text
// Cleaner: sort-independent top/bottom using RANK on both directions
Top:    RANK(SUM([Net Revenue]))
Bottom: RANK(SUM(-[Net Revenue]))
```

Trade-off: a rank table calculation depends on the view's addressing (here, Customer Name), so adding another dimension changes the ranking. If the ranking is a business definition (top 5 customers by revenue this quarter), compute it in SQL at the correct grain so every workbook agrees.

```sql
-- Warehouse-side top/bottom, deterministic and reusable
WITH ranked AS (
  SELECT customer_id, SUM(net_revenue) AS revenue,
         RANK() OVER (ORDER BY SUM(net_revenue) DESC) AS rnk_desc,
         RANK() OVER (ORDER BY SUM(net_revenue) ASC)  AS rnk_asc
  FROM analytics.mart_sales_daily
  WHERE date_day >= DATE '2026-07-01'
  GROUP BY customer_id
)
SELECT 'top' AS bucket, * FROM ranked WHERE rnk_desc <= 5
UNION ALL
SELECT 'bottom', * FROM ranked WHERE rnk_asc <= 5;
```

## What is the Rank Function in Tableau?

Mechanics: rank functions are table calculations, so they compute over the values currently in the view and follow the addressing and partitioning you set. `RANK` leaves gaps after ties, `RANK_DENSE` does not consume a position for a tie, `RANK_MODIFIED` assigns the same rank to a tie and continues at that rank, and `RANK_UNIQUE` gives every row a distinct rank. Common mistake: forgetting that adding a dimension to the view changes the addressing and therefore the ranking, which is why a rank meant as a business rule belongs in the warehouse.

Rank function is used to give positions (rank) to any measure in the data set. Tableau can rank measure in the following ways:
Rank: The rank function in Tableau accepts two arguments: aggregated measure and ranking order (optional) with a default value of desc.
Rank_dense: The rank_dense also accepts the two arguments: aggregated measure and ranking order. This assigns the same rank to the same values but doesn't stop there and keeps incrementing with the other values. For instance, if you have values 10, 20, 20, 30, then ranks will be 1, 2, 2, 3.
Rank_modified: The rank_modified assigns the same rank to similar values.
Rank_unique: The rank_unique assigns a unique rank to each and every value. For example, If the values are 10, 20, 20, 30 then the assigned ranks will be 1,2,3,4 respectively.

| Values | RANK | RANK_DENSE | RANK_MODIFIED | RANK_UNIQUE |
|---|---|---|---|---|
| 30 | 1 | 1 | 1 | 1 |
| 20 | 2 | 2 | 2 | 2 |
| 20 | 2 | 2 | 2 | 3 |
| 10 | 4 | 3 | 4 | 4 |

Rank functions are table calculations, not aggregations: they compute in the view and honor the addressing/partitioning you set. `RANK_DENSE` is the one to use for "top N categories" because ties do not consume a position, and `RANK` is the SQL-standard behavior with gaps after ties.

## What is the difference between Tableau and other similar tools like QlikView or IBM Cognos?

The current framing: Qlik uses an in-memory associative engine with a different governance model; Power BI competes on price and Microsoft integration; Looker competes on a code-defined semantic layer and embedded analytics. The decision criteria are reuse of metric definitions, embedding, cost per user, and whether applications must consume the same metrics as people, not chart aesthetics.

Tableau is different than QlikView or IBM Cognos for various reasons:
Tableau is an intuitive data visualization tool simplifying the story creation by simple drag and drop techniques. On the other hand, BI tools like QlikView or Cognos convert data into metadata to let the users explore data relations. If your presentation runs around presenting data in aesthetic visualizations then opt for Tableau. If not, and might need a full BI platform then go for Cognos/QlikView
The ease of use or extracting data details is easier in Tableau than compared to extensive BI tools like Cognos. With Tableau, your team members, be it a guy from sales can easily read the data and give insights. But with Cognos, only members with extensive tool knowledge are appreciated and welcomed.

A more current framing: Qlik uses an in-memory associative engine with a different governance model, Power BI competes on price and Microsoft integration, and Looker competes on a code-defined semantic layer (LookML) with embedded analytics. The decision criteria are semantic-layer reuse, embedding, cost per user, and whether metric definitions must be consumed by applications — not just chart aesthetics.

## How do you design an extract so it refreshes fast and stays small?

Extract design is a grain and filter problem before it is a storage problem. Materialize the columns the dashboards actually use, filter rows to the window that is queried, and pre-aggregate only when the drill-through can accept the coarser grain. The `.hyper` file stores each column in an encoded columnar form, so a string column costs a fixed-width reference per row plus a dictionary once, while numeric and date columns cost a fixed width per row. That is why dropping four unused string columns and capping the date window usually shrinks an extract far more than tuning compression.


```text
Extract design checklist
  rows      : extract filter on the date window and on the business partition
  columns   : data source filter / hide unused fields before extract
  grain     : aggregate to the dashboard grain when drill-through is not required
  refresh   : incremental with "Refresh if" [Order Date] > last refresh value
  monitoring: alert on refresh duration and failure, not on job started
```

Incremental refresh only works when the increment key is monotonic and the extract can identify new rows. A `created_at` or a date partition key works; an `updated_at` that is edited in place does not, because the extract never revisits old rows. If late-arriving edits matter, refresh the affected partitions with a bounded full refresh, or make the warehouse surface a key that changes with the row and treat it as a replace operation.

Common mistake: making the extract small enough to refresh quickly but too coarse for the questions the dashboard asks. Decide the grain from the questions first, then reduce size with filters and column pruning rather than by dropping dimensions.

In [ ]:
# Review-only: refresh an extract with tabcmd from a scheduler, then check the exit status.
# tabcmd is a separate CLI for Tableau Server/Cloud; it should run on a server, not on a laptop.
import subprocess

server = "https://tableau.example.com"
cmd = [
    "tabcmd", "refreshextracts",
    "--project", "Finance",
    "--datasource", "Certified Sales",
    "--server", server,
    "--site", "finance",
]
print(" ".join(cmd))
# subprocess.run(cmd, check=True, capture_output=True, text=True)


## How do you reduce query cost with data source filters, context filters, and pre-aggregation?

A data source filter is the cheapest lever because it narrows the query the source or Hyper must run, and it applies to both live and extract connections. A context filter is heavier: it creates a temporary filtered data set and forces every later filter to depend on it, which is exactly what you need when a FIXED LOD or a top-N set must respect a filter, but it adds a query pass and can slow an otherwise fast view. Pre-aggregation is the structural lever: one row per day per category instead of one row per order line means the view scans a much smaller table.

```sql
-- Pre-aggregate to the dashboard grain once, in the warehouse.
CREATE OR REPLACE TABLE analytics.agg_sales_daily AS
SELECT
  d.date_day,
  p.category,
  r.region,
  SUM(f.net_revenue) AS net_revenue,
  COUNT(DISTINCT f.order_id) AS orders
FROM analytics.fact_sales f
JOIN analytics.dim_date d    ON d.date_key = f.date_key
JOIN analytics.dim_product p ON p.product_key = f.product_key
JOIN analytics.dim_region r  ON r.region_key = f.region_key
GROUP BY 1, 2, 3;
```


Order the levers by cost: a data source filter costs nothing to add, a context filter costs one extra pass, and pre-aggregation costs a pipeline but pays off on every view. Reduce rows before you add compute. Common mistake: stacking several context filters, each of which builds and scans a temporary table, when a single data source filter on the same column would have pruned the data earlier and for free.

## How do you diagnose a slow workbook?

Performance Recording is the primary tool. It records a timeline per sheet and splits the time into query execution in the source, query execution in Hyper, computation and layout on the client, and rendering the marks. Read it by sheet and by the largest block: if the source query dominates, fix the data or push more work to the warehouse; if rendering dominates, reduce marks; if computing layout dominates, simplify the dashboard and its actions. Query Performance in Tableau Server and the View Performance report give the aggregate view across users.

A useful mental model is that a workbook is a factory with four stages, and the slowest stage sets the clock. SQL time is the warehouse or Hyper; compilation time is Tableau turning the viz into a query; layout time is placing marks; rendering time is drawing pixels. `tabjolt` or a REST-API concurrency probe tells you whether the slowness is per-view or only under load.


In [ ]:
# Review-only: bucket a performance recording by sheet and name the dominant stage.
# The same numbers appear in Server's View Performance report; the point is the classification.
recording = {
    "events": [
        {"sheet": "Revenue", "stage": "query", "seconds": 4.2},
        {"sheet": "Revenue", "stage": "render", "seconds": 0.3},
        {"sheet": "KPI", "stage": "query", "seconds": 0.1},
        {"sheet": "KPI", "stage": "render", "seconds": 1.8},
    ]
}

by_sheet = {}
for event in recording["events"]:
    by_sheet.setdefault(event["sheet"], {}).setdefault(event["stage"], 0.0)
    by_sheet[event["sheet"]][event["stage"]] += event["seconds"]

for sheet, stages in sorted(by_sheet.items()):
    bottleneck = max(stages, key=stages.get)
    print(f"{sheet}: {stages} -> bottleneck={bottleneck}")

assert by_sheet["Revenue"]["query"] == 4.2
assert max(by_sheet["KPI"], key=by_sheet["KPI"].get) == "render"


When the query stage dominates, move the work into the warehouse: pre-aggregate, materialize a view, or move the join there and connect to the result. When the render stage dominates, fix the view: fewer marks, fewer high-cardinality fields on Detail, and aggregation instead of a row-level query. Only when both are reasonable should you look at server capacity. Common mistake: adding nodes to the cluster to fix one slow sheet, when the recording shows a single table scan that an extract or a mart would remove.

## How do you implement row-level security without hurting performance?

Tableau has three layers. The warehouse is the strongest: row access policies or secure views mean the rule travels with the data and no extract can leak it. A Tableau user filter is the second layer: `[Region] = USERATTRIBUTE('region')` runs inside the query, so it also prunes rows on a live source. An entitlements table is the third and most flexible: land `user -> allowed dimension` mappings in the warehouse and relate or join them, so one filter serves many attributes. `USERNAME()` works when the account name matches the data; site user attributes are more robust because they are managed centrally.

```sql
-- Entitlements pattern: one row per user per dimension value they may see.
CREATE OR REPLACE TABLE analytics.user_region_access AS
SELECT 'alice@acme.com' AS user_name, 'EMEA' AS region
UNION ALL
SELECT 'bob@acme.com', 'US';
-- In Tableau, relate user_region_access to the fact on region and filter
-- [user_name] = USERNAME(), or apply the same predicate as a data source filter.
```


Performance depends on where the filter runs. A user filter on an indexed, low-cardinality column that matches the source's partitioning prunes early. A user filter on a high-cardinality or computed column scans first and filters later, which is painful on a live connection. On an extract, the user filter is applied by Hyper after the extract is built, so it must not be baked into an extract filter; an extract filter would remove rows for everyone. Common mistake: duplicating one workbook per region to enforce access, which multiplies maintenance and still leaks when someone republishes the source.

## Tableau exercises

These exercises use only the Python standard library and model the decisions in the chapter above with small, runnable arithmetic. Each one states the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are deliberately round so the check is on the formula rather than on a real workbook. Real runs add variance from filters, caching, and encoding, so the value is knowing which quantity to compute and which direction a design change moves it.

### Try it 1. Extract filter or data source filter?

**Task.** A source holds 10,000,000 rows across eight years. The executive dashboard needs the last two years (2,500,000 rows) and a live drill-through on the published source still needs every row. Choose between an extract filter and a data source filter for each consumer, then write the rule.

**Hint.** A data source filter narrows the query for everyone. An extract filter narrows only what is written into the `.hyper` file, so it is the choice when some other consumer still needs the full row set.

**Expected result.** The executive dashboard gets an `extract filter`; a workbook that needs all rows gets `no row filter`.

In [ ]:
SOURCE_ROWS = 10_000_000
ROWS_NEEDED = 2_500_000


def choose_row_filter(total_rows, needed_rows, live_needs_all_rows):
    if needed_rows == total_rows:
        return "no row filter"
    if live_needs_all_rows:
        return "extract filter"
    return "data source filter"


executive = choose_row_filter(SOURCE_ROWS, ROWS_NEEDED, live_needs_all_rows=True)
audit = choose_row_filter(SOURCE_ROWS, SOURCE_ROWS, live_needs_all_rows=False)
print("executive dashboard:", executive)
print("full-history workbook:", audit)

assert executive == "extract filter"
assert audit == "no row filter"


### Try it 2. Estimate the extract size

**Task.** Estimate the size of a Hyper extract for 10,000,000 rows and ten columns. Numeric and date columns cost 8 bytes per row. Each string column costs a 4-byte dictionary reference per row plus 12 bytes per distinct value in the dictionary once. Compare it with a naive fixed-width estimate of 8 bytes per column per row.

**Hint.** Keep per-row bytes and dictionary bytes in separate accumulators, then add them. A dictionary is tiny when a string column has few distinct values and expensive when it is nearly unique.

**Expected result.** The estimate is `775,480,144` bytes (about `0.72` GiB) against a naive `800,000,000` bytes.

In [ ]:
ROWS = 10_000_000
AVG_STRING_BYTES = 12
COLUMNS = [
    ("order_id", "string", 10_000_000),
    ("order_date", "date", 2_920),
    ("customer_id", "string", 1_200_000),
    ("product_id", "string", 90_000),
    ("region", "string", 12),
    ("unit_price", "numeric", 0),
    ("quantity", "numeric", 0),
    ("net_revenue", "numeric", 0),
    ("cost", "numeric", 0),
    ("margin", "numeric", 0),
]


def extract_bytes(rows, columns):
    per_row = 0
    dictionary = 0
    for _, kind, distinct in columns:
        if kind in ("numeric", "date"):
            per_row += 8
        elif kind == "string":
            per_row += 4
            dictionary += distinct * AVG_STRING_BYTES
    return rows * per_row + dictionary


size = extract_bytes(ROWS, COLUMNS)
naive = ROWS * len(COLUMNS) * 8
print(f"estimated extract = {size:,} bytes ({size / 1024 ** 3:.2f} GiB)")
print(f"naive fixed width = {naive:,} bytes ({naive / 1024 ** 3:.2f} GiB)")

assert size == 775_480_144
assert naive == 800_000_000
assert round(size / 1024 ** 3, 2) == 0.72


### Try it 3. Order the filters by evaluation cost

**Task.** Given six filters, order them by Tableau's order of operations, cheapest and earliest first. Only the positions of the six named types matter here; FIXED LOD sits between context and dimension filters in the full order.

**Hint.** Extract, then data source, then context, then dimension, then measure, then table calculation. Sort by that ranking rather than by the text.

**Expected result.** Extract, data source, context, dimension, measure, table calculation.

In [ ]:
FILTERS = [
    ("measure filter: SUM(margin) > 1000", "measure"),
    ("context filter: region IN (EMEA, US)", "context"),
    ("extract filter: order_date >= 2018-01-01", "extract"),
    ("table calculation filter: RANK(SUM(revenue)) <= 20", "table_calculation"),
    ("dimension filter: category = 'Bikes'", "dimension"),
    ("data source filter: exclude unused columns", "data_source"),
]
ORDER = {
    "extract": 1,
    "data_source": 2,
    "context": 3,
    "dimension": 5,
    "measure": 7,
    "table_calculation": 9,
}

ordered = [name for name, kind in sorted(FILTERS, key=lambda item: ORDER[item[1]])]
for position, name in enumerate(ordered, 1):
    print(f"{position}. {name}")

assert ordered[0].startswith("extract")
assert ordered[1].startswith("data source")
assert ordered[-1].startswith("table calculation")
assert ordered.index("context filter: region IN (EMEA, US)") < ordered.index(
    "dimension filter: category = 'Bikes'"
)


### Try it 4. Ratio of sums or average of ratios?

**Task.** Two regions have very different order counts. Compute the revenue per order as a ratio of sums and as an average of ratios, and decide which one a single LOD-based measure should return.

**Hint.** A ratio of sums is `SUM(revenue) / SUM(orders)`. An average of ratios is `AVG(revenue / orders)`. The first matches `SUM([Revenue]) / SUM([Orders])` in one view; the second is what you get when you average a row-level division in the view.

**Expected result.** `1810.0` per order for the ratio of sums and `1050.0` for the average of ratios.

In [ ]:
ROWS = [
    {"region": "East", "orders": 10, "revenue": 1_000},
    {"region": "West", "orders": 90, "revenue": 180_000},
]

ratio_of_sums = sum(r["revenue"] for r in ROWS) / sum(r["orders"] for r in ROWS)
average_of_ratios = sum(r["revenue"] / r["orders"] for r in ROWS) / len(ROWS)

print(f"SUM(revenue)/SUM(orders) = {ratio_of_sums:,.1f} per order")
print(f"AVG(revenue/orders)      = {average_of_ratios:,.1f} per order")

assert round(ratio_of_sums, 1) == 1810.0
assert round(average_of_ratios, 1) == 1050.0


### Try it 5. Plan an incremental refresh from a date key

**Task.** The last extract refresh captured rows through 2026-10-01. The source now has rows through 2026-10-07. Using `order_date` as the increment key, decide whether to refresh and how many days of data the incremental pass appends.

**Hint.** Compare the two dates as ISO strings or dates. If the source is ahead, the "Refresh if" condition `[order_date] > last refresh value` appends the difference in days.

**Expected result.** `incremental` with `6` days appended.

In [ ]:
from datetime import date

LAST_REFRESH_DATE = "2026-10-01"
SOURCE_MAX_DATE = "2026-10-07"
INCREMENT_KEY = "order_date"
DATE_FORMAT = "%Y-%m-%d"


def plan_refresh(last_date, source_max_date):
    if last_date >= source_max_date:
        return "no refresh", 0
    last = date.fromisoformat(last_date)
    current = date.fromisoformat(source_max_date)
    return "incremental", (current - last).days


mode, days = plan_refresh(LAST_REFRESH_DATE, SOURCE_MAX_DATE)
print(f"mode={mode} key={INCREMENT_KEY} days_to_append={days}")

assert mode == "incremental"
assert days == 6


### Try it 6. Size an aggregation from the grain

**Task.** A fact table holds 250,000,000 rows. Pre-aggregate it to one row per day per category per region, with 730 days, 40 categories, and 12 regions. Compute the aggregated row count and the reduction factor, capping the aggregate at the fact row count.

**Hint.** Multiply the three dimension cardinalities to get the grain size. Cap it with `min`, then divide the fact rows by the result.

**Expected result.** `350,400` aggregated rows, a reduction factor above `700`.

In [ ]:
FACT_ROWS = 250_000_000
DAYS = 730
CATEGORIES = 40
REGIONS = 12

grain_rows = DAYS * CATEGORIES * REGIONS
aggregated_rows = min(grain_rows, FACT_ROWS)
reduction = FACT_ROWS / aggregated_rows

print(f"fact rows        = {FACT_ROWS:,}")
print(f"aggregated rows  = {aggregated_rows:,}")
print(f"reduction factor = {reduction:,.0f}x")

assert aggregated_rows == 350_400
assert reduction > 700


### Try it 7. Apply a row-level security user filter

**Task.** Implement the Tableau user filter `USERATTRIBUTE('region') = 'ALL' OR [Region] = USERATTRIBUTE('region')` in Python and print the visible rows and revenue per user. An auditor with the `ALL` attribute sees every region.

**Hint.** Short-circuit on `ALL` first, then compare the row's region to the user's attribute.

**Expected result.** Alice sees one EMEA row, Bob sees one US row, and the auditor sees all three rows.

In [ ]:
USERS = {
    "alice@acme.com": "EMEA",
    "bob@acme.com": "US",
    "auditor@acme.com": "ALL",
}
ROWS = [("EMEA", 1200), ("US", 900), ("APAC", 300)]


def visible_region(region, user_attribute):
    return user_attribute == "ALL" or region == user_attribute


for user, attribute in USERS.items():
    visible = [row for row in ROWS if visible_region(row[0], attribute)]
    revenue = sum(row[1] for row in visible)
    print(f"{user:20s} attribute={attribute:5s} rows={len(visible)} revenue={revenue}")

assert visible_region("EMEA", "EMEA") is True
assert visible_region("US", "EMEA") is False
assert visible_region("APAC", "ALL") is True


## Real world use cases for Tableau

Across all five cases the same three decisions repeat: where the grain of truth lives (a warehouse mart, not a workbook), how fresh the data must be (live for minutes, extract for a review cadence), and who may see which rows (warehouse policy or a Tableau user filter). Answer those explicitly and the workbook design usually follows without debate.

### 1. Retail category management on a certified extract

Business context: a grocery retailer's category managers review sales, margin, and stock cover every Monday across 1,200 stores and 90,000 SKUs; the warehouse holds 4 years of line-item history (~2.1B rows, 380 GB). Managers need sub-5-second filtering.

```text
warehouse star schema (fact_sales + dims)
   -> nightly dbt mart mart_sales_daily (~250M rows)
   -> Tableau published data source "Certified Sales" (Hyper extract, 22 GB)
   -> parameter-driven dashboard: category, region, week, top-N
```

```text
// Dashboard parameter: [Week Offset] (integer, default 0)
// Calculated date: TODAY() - [Week Offset] * 7
```


Why this design: the extract is refreshed nightly after a 19:00 warehouse load, which matches the Monday review cadence while avoiding a live query against a busy warehouse; the certified published source means every manager sees the same margin definition, and the extract filter caps growth at two years so dashboard latency stays stable.

### 2. Fintech regulatory reporting with row-level security

Business context: a payments company must give regional compliance teams scoped access to transaction reporting (EU, UK, US) while auditors see everything. Data is 180M transactions/month with strict access requirements.

```text
regulated warehouse tables (transaction_fact, dim_region)
   -> live connection (no extract, so warehouse RLS applies at query time)
   -> Tableau user filters on [Region] = USERNAME()/USERATTRIBUTE('region')
   -> subscriptions to compliance teams with PDF snapshots archived
```

```sql
-- Warehouse-side row-level security is the source of truth
CREATE OR REPLACE ROW ACCESS POLICY rap_region
ON analytics.transaction_fact
GRANT TO ('role:compliance_reader')
FILTER USING (region = CURRENT_ROLE());
```

```text
// Tableau user filter (defense in depth, for extract-based sources)
[Region] = USERATTRIBUTE('region')
// USERATTRIBUTE requires the site to be configured with the attribute;
// auditors get 'ALL' and the calculation short-circuits:
USERATTRIBUTE('region') = 'ALL' OR [Region] = USERATTRIBUTE('region')
```

Why this design: compliance data should not be copied into a movable extract, so the connection stays live and the warehouse enforces access; the Tableau user filter is a second layer for any extract and prevents an accidental cross-region leak if a source is republished.

### 3. Marketing spend efficiency across channels

Business context: a D2C brand spends $1.8M/month across six ad platforms and wants CAC and ROAS by channel and campaign, updated daily by 08:00. Volume: 90M impressions/day, 14M sessions/day, 220K orders/day, six API feeds landed in the warehouse.

```text
ad platform APIs -> warehouse staging -> mart_channel_daily (spend, clicks, orders, revenue)
   -> Tableau dashboard: CAC = spend / new_customers, ROAS = revenue / spend
   -> parameter [Date Range] + filter action from channel to campaign detail
```

```text
// CAC with an LOD so it is correct at every level of the view
SUM([Spend]) / NULLIF(COUNTD([New Customer ID]), 0)
```


Why this design: spend and revenue live in one warehouse mart with a shared date and channel key, so CAC is a true ratio rather than a ratio of averages; the parameterized date range lets media buyers compare 7/28/90-day windows without six separate workbooks.

### 4. SaaS product adoption for customer success

Business context: a B2B SaaS company's CSMs need account-level usage, seat utilization, and risk flags, with each CSM seeing only their book of business. Volume: 340 enterprise accounts, 900M product events/month, refreshed hourly.

```text
product events -> warehouse -> mart_account_usage_hourly
   -> live connection with Tableau user filter on [CSM Email] = USERNAME()
   -> account scorecard + risk heatmap + drill-through to feature usage
```

```text
// Seat utilization LOD: independent of the feature rows in the view
{ FIXED [Account ID] : SUM([Active Seats]) } / NULLIF({ FIXED [Account ID] : MAX([Licensed Seats]) }, 0)
```


Why this design: the CSM user filter keeps each rep scoped without duplicating dashboards per region, and FIXED LODs compute account-level ratios that stay correct whether the CSM is looking at accounts, features, or weeks.

### 5. Operations monitoring for a logistics network

Business context: a 3PL monitors on-time delivery across 60 hubs; dispatch needs a live view during peak, and planning needs a trend view. Volume: 1.4M shipments/day, 40M scan events/day.

```text
scanner events -> Kafka -> warehouse micro-batch (5 min) -> mart_shipment_status
   -> live connection dashboard (on-time %, delays, hub backlog)
   -> hyper extract for the 12-month trend workbook (refreshed daily 03:00)
```

```text
// On-time % as a ratio of sums, not an average of averages
SUM([On Time Shipments]) / NULLIF(SUM([Total Shipments]), 0)
```


Why this design: the operational dashboard uses a live connection so a 5-minute-old micro-batch is visible immediately, while the heavy 12-month trend uses an extract so it does not compete with dispatch queries; splitting the two workloads by connection type keeps both fast without extra server capacity.